<a href="https://colab.research.google.com/github/kireetinotkirti/NETFLIC-DATA-ANALYSIS-USING-GOOGLE-COLAB/blob/main/NTFXDataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""

  Netflix Data Analysis — Full EDA Pipeline
  Dataset: 6,234 titles | Python · Pandas · Seaborn


What this script does:
  1. Loads and cleans the Netflix dataset (handles missing values, types)
  2. Performs Exploratory Data Analysis (EDA)
  3. Generates 10 publication-quality visualizations
  4. Prints key insights recruiters want to see

Author  : Kireeti Dodla
Dataset : netflix_titles.csv (Kaggle / generated sample)
Tools   : Python 3, Pandas, Seaborn, Matplotlib, NumPy
"""


'\n\n  Netflix Data Analysis — Full EDA Pipeline            \n  Dataset: 6,234 titles | Python · Pandas · Seaborn        \n\n\nWhat this script does:\n  1. Loads and cleans the Netflix dataset (handles missing values, types)\n  2. Performs Exploratory Data Analysis (EDA)\n  3. Generates 10 publication-quality visualizations\n  4. Prints key insights recruiters want to see\n\nAuthor  : Kireeti Dodla\nDataset : netflix_titles.csv (Kaggle / generated sample)\nTools   : Python 3, Pandas, Seaborn, Matplotlib, NumPy\n'

In [ ]:

# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
import sys
from collections import Counter

warnings.filterwarnings('ignore')


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATASET_PATH  = "/content/netflix_titles.csv"
OUTPUT_DIR    = "outputs"
FIGURE_DPI    = 150

# Netflix brand palette
NETFLIX_RED   = "#E50914"
NETFLIX_DARK  = "#141414"
NETFLIX_GRAY  = "#564d4d"
PALETTE_MAIN  = [NETFLIX_RED, "#B81D24", "#F5F5F1", "#831010",
                 "#E87C03", "#F5C518", "#4ECDC4", "#45B7D1",
                 "#96CEB4", "#FFEAA7", "#DDA0DD", "#98D8C8"]

sns.set_theme(style="darkgrid", palette=PALETTE_MAIN)
plt.rcParams.update({
    "figure.facecolor":  NETFLIX_DARK,
    "axes.facecolor":    "#1a1a1a",
    "axes.labelcolor":   "#F5F5F1",
    "axes.titlecolor":   "#F5F5F1",
    "xtick.color":       "#aaaaaa",
    "ytick.color":       "#aaaaaa",
    "grid.color":        "#333333",
    "grid.linewidth":    0.5,
    "text.color":        "#F5F5F1",
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.spines.left":  False,
    "axes.spines.bottom":False,
})

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — DATA LOADING & CLEANING
# ─────────────────────────────────────────────────────────────────────────────
def load_and_clean(path: str) -> pd.DataFrame:
    """
    Load and clean the Netflix dataset.
    Handles: date parsing, missing values, feature engineering.
    """
    print("\n" + "═"*60)
    print("  📂  LOADING DATASET")
    print("═"*60)

    if not os.path.exists(path):
        print(f"  ❌  File '{path}' not found.")
        print("  ℹ   Download from: https://www.kaggle.com/datasets/shivamb/netflix-shows")
        sys.exit(1)

    df = pd.read_csv(path)
    print(f"  ✅  Loaded  : {df.shape[0]:,} rows × {df.shape[1]} columns")

    # ── Missing value audit ──────────────────────────────────────────────────
    print("\n  📊  Missing Value Audit (before cleaning):")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    for col in missing[missing > 0].index:
        print(f"       {col:<15} {missing[col]:>5} missing  ({missing_pct[col]}%)")

    # ── Clean date_added ─────────────────────────────────────────────────────
    df['date_added']  = pd.to_datetime(df['date_added'], errors='coerce')
    df['year_added']  = df['date_added'].dt.year
    df['month_added'] = df['date_added'].dt.month
    df['month_name']  = df['date_added'].dt.strftime('%b')

    # ── Fill categorical missing values ──────────────────────────────────────
    fill_map = {
        'director': 'Unknown',
        'cast':     'Unknown',
        'country':  'Unknown',
        'rating':   'NR',
    }
    for col, fill in fill_map.items():
        df[col].fillna(fill, inplace=True)

    # ── Extract numeric duration ──────────────────────────────────────────────
    df['duration_int'] = df['duration'].str.extract(r'(\d+)').astype(float)

    # ── Separate movies vs TV shows ───────────────────────────────────────────
    df['is_movie']   = df['type'] == 'Movie'
    df['is_tvshow']  = df['type'] == 'TV Show'

    # ── Years since release ──────────────────────────────────────────────────
    current_year = 2024
    df['age_years'] = current_year - df['release_year']

    # ── Primary genre (first listed genre) ───────────────────────────────────
    df['primary_genre'] = df['listed_in'].str.split(',').str[0].str.strip()

    # ── Primary country ──────────────────────────────────────────────────────
    df['primary_country'] = df['country'].str.split(',').str[0].str.strip()

    print("\n  ✅  Data cleaned successfully.")
    print(f"  📐  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    return df

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — EDA & INSIGHTS
# ─────────────────────────────────────────────────────────────────────────────
def print_insights(df: pd.DataFrame):
    print("\n" + "═"*60)
    print("  🔍  KEY INSIGHTS")
    print("═"*60)

    movies = df[df['type'] == 'Movie']
    shows  = df[df['type'] == 'TV Show']

    print(f"\n  Content Split:")
    print(f"    Movies   : {len(movies):>5,}  ({len(movies)/len(df)*100:.1f}%)")
    print(f"    TV Shows : {len(shows):>5,}  ({len(shows)/len(df)*100:.1f}%)")

    print(f"\n  Top 5 Genres (Movies):")
    movie_genres = pd.Series([g.strip() for genres in movies['listed_in']
                               for g in genres.split(',')]).value_counts().head(5)
    for genre, cnt in movie_genres.items():
        print(f"    {genre:<30} {cnt:>5,}")

    print(f"\n  Top 5 Genres (TV Shows):")
    tv_genres = pd.Series([g.strip() for genres in shows['listed_in']
                            for g in genres.split(',')]).value_counts().head(5)
    for genre, cnt in tv_genres.items():
        print(f"    {genre:<30} {cnt:>5,}")

    print(f"\n  Average Movie Duration : {movies['duration_int'].mean():.0f} min")
    print(f"  Average TV Show Seasons: {shows['duration_int'].mean():.1f} seasons")

    print(f"\n  Most Common Ratings:")
    for rating, cnt in df['rating'].value_counts().head(5).items():
        pct = cnt / len(df) * 100
        bar = "█" * int(pct)
        print(f"    {rating:<8} {cnt:>5,}  {bar} {pct:.1f}%")

    peak_year = df.groupby('year_added').size().idxmax()
    print(f"\n  Peak Year of Content Added: {int(peak_year)}")
    print(f"  Oldest Title Release Year : {df['release_year'].min()}")
    print(f"  Newest Title Release Year : {df['release_year'].max()}")

    top_country = df['primary_country'].value_counts().index[0]
    print(f"\n  #1 Content-Producing Country: {top_country}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — VISUALIZATIONS
# ─────────────────────────────────────────────────────────────────────────────
def save_fig(name: str):
    path = os.path.join(OUTPUT_DIR, name)
    plt.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight',
                facecolor=plt.gcf().get_facecolor())
    plt.close()
    print(f"  💾  Saved → {path}")


# ── Plot 1 : Movies vs TV Shows (Donut Chart) ─────────────────────────────────
def plot_content_type(df):
    type_counts = df['type'].value_counts()
    fig, ax = plt.subplots(figsize=(7, 7), facecolor=NETFLIX_DARK)
    wedge_props = {'width': 0.5, 'edgecolor': NETFLIX_DARK, 'linewidth': 3}
    wedges, texts, autotexts = ax.pie(
        type_counts, labels=type_counts.index,
        autopct='%1.1f%%', startangle=90,
        colors=[NETFLIX_RED, "#4ECDC4"],
        wedgeprops=wedge_props, textprops={'color': '#F5F5F1', 'fontsize': 14}
    )
    for at in autotexts:
        at.set_fontsize(13)
        at.set_fontweight('bold')
    # Center text
    ax.text(0, 0, f"{len(df):,}\nTitles", ha='center', va='center',
            fontsize=16, fontweight='bold', color='#F5F5F1')
    ax.set_title("Movies vs TV Shows", fontsize=18, fontweight='bold',
                 color='#F5F5F1', pad=20)
    save_fig("01_content_type_split.png")


# ── Plot 2 : Top 15 Genres (Horizontal bar) ────────────────────────────────────
def plot_top_genres(df):
    all_genres = pd.Series([g.strip() for gs in df['listed_in']
                             for g in gs.split(',')]).value_counts().head(15)
    fig, ax = plt.subplots(figsize=(11, 7), facecolor=NETFLIX_DARK)
    colors = [NETFLIX_RED if i == 0 else "#B81D24" if i < 3 else "#564d4d"
              for i in range(len(all_genres))]
    bars = ax.barh(all_genres.index[::-1], all_genres.values[::-1],
                   color=colors[::-1], edgecolor='none', height=0.7)
    for bar, val in zip(bars, all_genres.values[::-1]):
        ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height()/2,
                f"{val:,}", va='center', ha='left', color='#aaaaaa', fontsize=9)
    ax.set_xlabel("Number of Titles", labelpad=10)
    ax.set_title("Top 15 Genres on Netflix", fontsize=16, fontweight='bold',
                 color='#F5F5F1', pad=15)
    ax.tick_params(axis='y', labelsize=10)
    save_fig("02_top_genres.png")


# ── Plot 3 : Content Added Per Year (Area Chart) ──────────────────────────────
def plot_content_per_year(df):
    yearly = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
    yearly = yearly[yearly.index >= 2014]
    fig, ax = plt.subplots(figsize=(12, 6), facecolor=NETFLIX_DARK)
    ax.fill_between(yearly.index, yearly.get('Movie', 0),
                    alpha=0.8, color=NETFLIX_RED, label='Movies')
    ax.fill_between(yearly.index, yearly.get('TV Show', 0),
                    alpha=0.7, color='#4ECDC4', label='TV Shows')
    ax.plot(yearly.index, yearly.get('Movie', 0),
            color=NETFLIX_RED, linewidth=2)
    ax.plot(yearly.index, yearly.get('TV Show', 0),
            color='#4ECDC4', linewidth=2)
    ax.set_xlabel("Year", labelpad=10)
    ax.set_ylabel("Titles Added", labelpad=10)
    ax.set_title("Netflix Content Added Per Year", fontsize=16,
                 fontweight='bold', color='#F5F5F1', pad=15)
    ax.legend(facecolor='#1a1a1a', labelcolor='#F5F5F1', fontsize=11)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    save_fig("03_content_per_year.png")


# ── Plot 4 : Ratings Distribution ────────────────────────────────────────────
def plot_ratings(df):
    rating_order = df['rating'].value_counts().index.tolist()
    fig, ax = plt.subplots(figsize=(12, 6), facecolor=NETFLIX_DARK)
    palette_r = sns.color_palette([NETFLIX_RED, "#B81D24", "#E87C03",
                                    "#F5C518", "#4ECDC4", "#45B7D1",
                                    "#96CEB4", "#FFEAA7", "#DDA0DD",
                                    "#98D8C8", "#aaaaaa", "#666666"],
                                   n_colors=len(rating_order))
    sns.countplot(data=df, x='rating', order=rating_order,
                  palette=palette_r, ax=ax, edgecolor='none')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}',
                    (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=9, color='#aaaaaa')
    ax.set_xlabel("Content Rating", labelpad=10)
    ax.set_ylabel("Number of Titles", labelpad=10)
    ax.set_title("Content Ratings Distribution", fontsize=16,
                 fontweight='bold', color='#F5F5F1', pad=15)
    save_fig("04_ratings_distribution.png")


# ── Plot 5 : Top 10 Countries ─────────────────────────────────────────────────
def plot_top_countries(df):
    country_counts = df[df['primary_country'] != 'Unknown'] \
                         ['primary_country'].value_counts().head(10)
    fig, ax = plt.subplots(figsize=(10, 7), facecolor=NETFLIX_DARK)
    colors = [NETFLIX_RED if i == 0 else
              "#B81D24" if i < 3 else "#564d4d"
              for i in range(10)]
    bars = ax.barh(country_counts.index[::-1], country_counts.values[::-1],
                   color=colors[::-1], edgecolor='none', height=0.65)
    for bar, val in zip(bars, country_counts.values[::-1]):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                f"{val:,}", va='center', fontsize=10, color='#aaaaaa')
    ax.set_xlabel("Number of Titles", labelpad=10)
    ax.set_title("Top 10 Content-Producing Countries", fontsize=16,
                 fontweight='bold', color='#F5F5F1', pad=15)
    save_fig("05_top_countries.png")


# ── Plot 6 : Movie Duration Distribution ──────────────────────────────────────
def plot_movie_duration(df):
    movies = df[(df['type'] == 'Movie') & df['duration_int'].notna()]
    movies = movies[movies['duration_int'].between(30, 250)]
    fig, ax = plt.subplots(figsize=(11, 6), facecolor=NETFLIX_DARK)
    ax.hist(movies['duration_int'], bins=40, color=NETFLIX_RED,
            edgecolor=NETFLIX_DARK, alpha=0.9)
    median_dur = movies['duration_int'].median()
    ax.axvline(median_dur, color='#F5C518', linewidth=2, linestyle='--',
               label=f'Median: {int(median_dur)} min')
    mean_dur = movies['duration_int'].mean()
    ax.axvline(mean_dur, color='#4ECDC4', linewidth=2, linestyle='--',
               label=f'Mean: {int(mean_dur)} min')
    ax.set_xlabel("Duration (minutes)", labelpad=10)
    ax.set_ylabel("Number of Movies", labelpad=10)
    ax.set_title("Movie Duration Distribution", fontsize=16,
                 fontweight='bold', color='#F5F5F1', pad=15)
    ax.legend(facecolor='#1a1a1a', labelcolor='#F5F5F1', fontsize=11)
    save_fig("06_movie_duration.png")


# ── Plot 7 : Content Added by Month (Heatmap) ─────────────────────────────────
def plot_monthly_heatmap(df):
    heat_df = df.dropna(subset=['year_added', 'month_added'])
    heat_df = heat_df[heat_df['year_added'] >= 2015]
    pivot = heat_df.pivot_table(index='month_added', columns='year_added',
                                 values='show_id', aggfunc='count', fill_value=0)
    pivot.index = ['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot)]
    fig, ax = plt.subplots(figsize=(14, 7), facecolor=NETFLIX_DARK)
    sns.heatmap(pivot, cmap='Reds', ax=ax, linewidths=0.5,
                linecolor='#141414', annot=True, fmt='d',
                annot_kws={'size': 9, 'color': '#F5F5F1'},
                cbar_kws={'label': 'Titles Added'})
    ax.set_title("Content Added by Month & Year", fontsize=16,
                 fontweight='bold', color='#F5F5F1', pad=15)
    ax.set_xlabel("Year", labelpad=10)
    ax.set_ylabel("Month", labelpad=10)
    ax.tick_params(colors='#aaaaaa')
    save_fig("07_monthly_heatmap.png")


# ── Plot 8 : Release Year Trend ────────────────────────────────────────────────
def plot_release_year(df):
    yearly = df[df['release_year'] >= 1980] \
               .groupby(['release_year','type']).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(13, 6), facecolor=NETFLIX_DARK)
    ax.fill_between(yearly.index, yearly.get('Movie', 0),
                    alpha=0.75, color=NETFLIX_RED, label='Movies')
    ax.fill_between(yearly.index, yearly.get('TV Show', 0),
                    alpha=0.6, color='#4ECDC4', label='TV Shows')
    ax.set_xlabel("Release Year", labelpad=10)
    ax.set_ylabel("Number of Titles", labelpad=10)
    ax.set_title("Content by Original Release Year (1980–Present)",
                 fontsize=16, fontweight='bold', color='#F5F5F1', pad=15)
    ax.legend(facecolor='#1a1a1a', labelcolor='#F5F5F1', fontsize=11)
    save_fig("08_release_year_trend.png")


# ── Plot 9 : TV Show Seasons Distribution ─────────────────────────────────────
def plot_tv_seasons(df):
    shows = df[(df['type'] == 'TV Show') & df['duration_int'].notna()]
    shows = shows[shows['duration_int'] <= 15]
    season_counts = shows['duration_int'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(10, 6), facecolor=NETFLIX_DARK)
    bars = ax.bar(season_counts.index.astype(int), season_counts.values,
                  color=[NETFLIX_RED if i == 0 else "#B81D24" if i < 2 else "#564d4d"
                          for i in range(len(season_counts))],
                  edgecolor='none', width=0.7)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())),
                ha='center', va='bottom', fontsize=9, color='#aaaaaa')
    ax.set_xlabel("Number of Seasons", labelpad=10)
    ax.set_ylabel("Number of TV Shows", labelpad=10)
    ax.set_title("TV Show Seasons Distribution", fontsize=16,
                 fontweight='bold', color='#F5F5F1', pad=15)
    save_fig("09_tv_seasons.png")


# ── Plot 10 : Genre Heatmap (Movies vs TV Shows) ──────────────────────────────
def plot_genre_type_heatmap(df):
    rows = []
    for _, row in df.iterrows():
        for genre in row['listed_in'].split(','):
            rows.append({'genre': genre.strip(), 'type': row['type']})
    genre_df = pd.DataFrame(rows)

    if genre_df.empty:
        print("  ℹ️  Skipping Genre Heatmap: No valid genre data found.")
        return

    top_genres = genre_df['genre'].value_counts().head(12).index
    genre_df = genre_df[genre_df['genre'].isin(top_genres)]

    if genre_df.empty:
        print("  ℹ️  Skipping Genre Heatmap: No data after filtering for top genres.")
        return

    pivot = genre_df.pivot_table(index='genre', columns='type',
                                  values='genre', aggfunc='count', fill_value=0)

    if pivot.empty:
        print("  ℹ️  Skipping Genre Heatmap: Pivot table is empty.")
        return

    fig, ax = plt.subplots(figsize=(8, 8), facecolor=NETFLIX_DARK)
    sns.heatmap(pivot, cmap='Reds', ax=ax, annot=True, fmt='d',
                linewidths=0.5, linecolor='#141414',
                annot_kws={'size': 11, 'color': '#F5F5F1'},
                cbar_kws={'label': 'Count'})
    ax.set_title("Genre Distribution: Movies vs TV Shows",
                 fontsize=15, fontweight='bold', color='#F5F5F1', pad=15)
    ax.set_xlabel("Type", labelpad=10)
    ax.set_ylabel("Genre", labelpad=10)
    ax.tick_params(colors='#aaaaaa')
    save_fig("10_genre_type_heatmap.png")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — DASHBOARD (Summary 2×2 grid)
# ─────────────────────────────────────────────────────────────────────────────
def plot_dashboard(df):
    fig = plt.figure(figsize=(18, 12), facecolor=NETFLIX_DARK)
    fig.suptitle("🎬  Netflix Content Analysis Dashboard",
                 fontsize=22, fontweight='bold', color='#F5F5F1', y=0.98)

    # ── Subplot 1: Content Split Donut ──────────────────────────────────────
    ax1 = fig.add_subplot(2, 3, 1)
    type_counts = df['type'].value_counts()
    wedge_props = {'width': 0.5, 'edgecolor': NETFLIX_DARK, 'linewidth': 2}
    ax1.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
            colors=[NETFLIX_RED, '#4ECDC4'], wedgeprops=wedge_props,
            textprops={'color': '#F5F5F1', 'fontsize': 11}, startangle=90)
    ax1.set_title("Content Split", fontweight='bold')
    ax1.text(0, 0, f"{len(df):,}\nTitles", ha='center', va='center',
             fontsize=12, fontweight='bold', color='#F5F5F1')

    # ── Subplot 2: Top 10 Genres ────────────────────────────────────────────
    ax2 = fig.add_subplot(2, 3, 2)
    all_genres = pd.Series([g.strip() for gs in df['listed_in']
                             for g in gs.split(',')]).value_counts().head(10)
    colors_g = [NETFLIX_RED if i < 2 else "#B81D24" if i < 5 else "#564d4d"
                for i in range(10)]
    ax2.barh(all_genres.index[::-1], all_genres.values[::-1],
             color=colors_g[::-1], edgecolor='none', height=0.65)
    ax2.set_title("Top 10 Genres", fontweight='bold')
    ax2.tick_params(axis='y', labelsize=9)

    # ── Subplot 3: Ratings ──────────────────────────────────────────────────
    ax3 = fig.add_subplot(2, 3, 3)
    top_ratings = df['rating'].value_counts().head(8)
    ax3.bar(top_ratings.index, top_ratings.values,
            color=[NETFLIX_RED, "#B81D24", "#E87C03", "#F5C518",
                   "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"][:len(top_ratings)],
            edgecolor='none')
    ax3.set_title("Ratings Distribution", fontweight='bold')
    ax3.tick_params(axis='x', rotation=45, labelsize=9)

    # ── Subplot 4: Content per Year ─────────────────────────────────────────
    ax4 = fig.add_subplot(2, 3, 4)
    yearly = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
    yearly = yearly[yearly.index >= 2015]
    ax4.fill_between(yearly.index, yearly.get('Movie', 0),
                     alpha=0.8, color=NETFLIX_RED, label='Movies')
    ax4.fill_between(yearly.index, yearly.get('TV Show', 0),
                     alpha=0.7, color='#4ECDC4', label='TV Shows')
    ax4.set_title("Content Added Per Year", fontweight='bold')
    ax4.legend(facecolor='#1a1a1a', labelcolor='#F5F5F1', fontsize=8)
    ax4.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # ── Subplot 5: Top Countries ────────────────────────────────────────────
    ax5 = fig.add_subplot(2, 3, 5)
    country_counts = df[df['primary_country'] != 'Unknown'] \
                         ['primary_country'].value_counts().head(8)
    ax5.barh(country_counts.index[::-1], country_counts.values[::-1],
             color=[NETFLIX_RED if i < 2 else "#B81D24" if i < 4 else "#564d4d"
                    for i in range(8)][::-1],
             edgecolor='none', height=0.6)
    ax5.set_title("Top 8 Countries", fontweight='bold')
    ax5.tick_params(axis='y', labelsize=9)

    # ── Subplot 6: Movie Duration Histogram ─────────────────────────────────
    ax6 = fig.add_subplot(2, 3, 6)
    movies = df[(df['type'] == 'Movie') & df['duration_int'].notna()]
    movies = movies[movies['duration_int'].between(30, 250)]
    ax6.hist(movies['duration_int'], bins=30, color=NETFLIX_RED,
             edgecolor=NETFLIX_DARK, alpha=0.9)
    ax6.axvline(movies['duration_int'].median(), color='#F5C518',
                linewidth=2, linestyle='--', label='Median')
    ax6.set_title("Movie Duration (min)", fontweight='bold')
    ax6.legend(facecolor='#1a1a1a', labelcolor='#F5F5F1', fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    save_fig("00_dashboard.png")
    print("\n  🎯  Dashboard saved → outputs/00_dashboard.png")

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("\n" + "█"*60)
    print("  🎬  Netflix Data Analysis — Starting Pipeline")
    print("█"*60)

    # 1. Load & Clean
    df = load_and_clean(DATASET_PATH)

    # 2. Print insights
    print_insights(df)

    # 3. Generate plots
    print("\n" + "═"*60)
    print("  📊  GENERATING VISUALIZATIONS")
    print("═"*60)

    plot_content_type(df)
    plot_top_genres(df)
    plot_content_per_year(df)
    plot_ratings(df)
    plot_top_countries(df)
    plot_movie_duration(df)
    plot_monthly_heatmap(df)
    plot_release_year(df)
    plot_tv_seasons(df)
    plot_genre_type_heatmap(df) # This function call is causing the error
    plot_dashboard(df)

    print("\n" + "═"*60)
    print(f"  ✅  All done! {len(os.listdir(OUTPUT_DIR))} files saved to ./{OUTPUT_DIR}/")
    print("═"*60 + "\n")


if __name__ == "__main__":
    main()



████████████████████████████████████████████████████████████
  🎬  Netflix Data Analysis — Starting Pipeline
████████████████████████████████████████████████████████████

════════════════════════════════════════════════════════════
  📂  LOADING DATASET
════════════════════════════════════════════════════════════
  ✅  Loaded  : 8,807 rows × 12 columns

  📊  Missing Value Audit (before cleaning):
       director         2634 missing  (29.9%)
       cast              825 missing  (9.4%)
       country           831 missing  (9.4%)
       date_added         10 missing  (0.1%)
       rating              4 missing  (0.0%)
       duration            3 missing  (0.0%)

  ✅  Data cleaned successfully.
  📐  Final shape: 8,807 rows × 21 columns

════════════════════════════════════════════════════════════
  🔍  KEY INSIGHTS
════════════════════════════════════════════════════════════

  Content Split:
    Movies   : 6,131  (69.6%)
    TV Shows : 2,676  (30.4%)

  Top 5 Genres (Movies):
    Interna